# ISY503 Assessment 3 — NLP Sentiment Classification

**Project:** ReviewLens  
**Group members:** Add names and student IDs before submission.

This notebook trains a neural network to classify English Amazon reviews as **Positive review** or **Negative review**. It uses the official Johns Hopkins Multi-Domain Sentiment Dataset.

## 1. Imports and reproducibility
The reusable implementation is kept in `ml/train_model.py` so the notebook and website use the same preprocessing and model architecture.

In [1]:
from pathlib import Path
import json
import random
import sys

import torch
from torch import nn
from torch.utils.data import DataLoader

sys.path.append(str(Path('ml').resolve()))
from train_model import (
    SEED, ReviewDataset, SentimentNetwork, build_vocabulary,
    class_counts, download_dataset, evaluate, export_browser_model,
    load_reviews, remove_length_outliers, stratified_split, train,
)

random.seed(SEED)
torch.manual_seed(SEED)
print('PyTorch version:', torch.__version__)

PyTorch version: 2.13.0+cpu


## 2. Load positive and negative reviews
The dataset contains books, DVDs, electronics, and kitchen products. Each domain has 1,000 positive and 1,000 negative labelled reviews.

In [2]:
dataset_root = download_dataset(Path('ml/data/raw'))
reviews_after_deduplication, duplicates_removed = load_reviews(dataset_root)
source_count = len(reviews_after_deduplication) + duplicates_removed

print('Source reviews:', source_count)
print('Exact duplicates removed:', duplicates_removed)
print('Remaining class counts:', class_counts(reviews_after_deduplication))

Source reviews: 8000
Exact duplicates removed: 148
Remaining class counts: {'negative': 3881, 'positive': 3971}


## 3. Cleaning, outlier removal, and data split
The cleaning pipeline normalises case, HTML, URLs, punctuation, contractions and whitespace. Exact duplicate removal is performed before the split to reduce leakage. Reviews outside the 1st–99th token-length percentiles are treated as outliers. A stratified 70/15/15 split keeps both sentiment classes represented.

In [3]:
reviews, outlier_summary = remove_length_outliers(reviews_after_deduplication)
train_reviews, validation_reviews, test_reviews = stratified_split(reviews)

print(outlier_summary)
print('Train:', len(train_reviews), class_counts(train_reviews))
print('Validation:', len(validation_reviews), class_counts(validation_reviews))
print('Test:', len(test_reviews), class_counts(test_reviews))

{'method': 'Removed reviews outside the 1st-99th token-length percentiles', 'minimum_tokens': 13, 'maximum_tokens': 749, 'removed': 154}
Train: 5388 {'negative': 2668, 'positive': 2720}
Validation: 1153 {'negative': 571, 'positive': 582}
Test: 1157 {'negative': 573, 'positive': 584}


## 4. Word encoding, padding, and batches
The vocabulary is fitted on training data only. Reviews are encoded using unigrams and adjacent bigrams. Every sequence is padded or truncated to 260 token IDs.

In [4]:
VOCABULARY_SIZE = 12_000
MAX_LENGTH = 260
BATCH_SIZE = 128

vocabulary = build_vocabulary(train_reviews, VOCABULARY_SIZE)
train_dataset = ReviewDataset(train_reviews, vocabulary, MAX_LENGTH)
validation_dataset = ReviewDataset(validation_reviews, vocabulary, MAX_LENGTH)
test_dataset = ReviewDataset(test_reviews, vocabulary, MAX_LENGTH)

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=generator)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print('Vocabulary size:', len(vocabulary))
print('One padded sequence shape:', train_dataset[0][0].shape)

Vocabulary size: 12000
One padded sequence shape: torch.Size([260])


## 5. Neural network architecture
The network learns 48-dimensional word embeddings, averages non-padding embeddings, applies a 64-unit dense layer with ReLU, uses 35% dropout during training, and returns one sentiment logit.

In [5]:
model = SentimentNetwork(
    vocabulary_size=len(vocabulary),
    embedding_dim=48,
    hidden_dim=64,
    dropout=0.35,
)
print(model)

SentimentNetwork(
  (embedding): Embedding(12000, 48, padding_idx=0)
  (hidden): Linear(in_features=48, out_features=64, bias=True)
  (dropout): Dropout(p=0.35, inplace=False)
  (output): Linear(in_features=64, out_features=1, bias=True)
)


## 6. Train with validation and early stopping
Training uses binary cross-entropy, AdamW, gradient clipping and early stopping. The best validation-loss checkpoint is restored before testing.

In [6]:
history = train(
    model,
    train_loader,
    validation_loader,
    epochs=24,
    learning_rate=0.003,
    checkpoint=Path('ml/artifacts/best_model.pt'),
)

Epoch 01 | train loss 0.6883 | validation loss 0.6753 | validation accuracy 64.27%


Epoch 02 | train loss 0.6498 | validation loss 0.5999 | validation accuracy 71.38%


Epoch 03 | train loss 0.5260 | validation loss 0.4955 | validation accuracy 75.37%


Epoch 04 | train loss 0.4017 | validation loss 0.4564 | validation accuracy 77.97%


Epoch 05 | train loss 0.3079 | validation loss 0.4395 | validation accuracy 78.58%


Epoch 06 | train loss 0.2393 | validation loss 0.4456 | validation accuracy 79.10%


Epoch 07 | train loss 0.1746 | validation loss 0.4614 | validation accuracy 79.97%


Epoch 08 | train loss 0.1235 | validation loss 0.4920 | validation accuracy 79.36%


Epoch 09 | train loss 0.0837 | validation loss 0.5338 | validation accuracy 78.75%
Early stopping: validation loss stopped improving.


## 7. Test-set evaluation
The main measures are accuracy, macro F1, ROC-AUC, class-specific recall and the confusion matrix. The test set is used only after model selection.

In [7]:
test_loss, test_metrics, _ = evaluate(model, test_loader, nn.BCEWithLogitsLoss())
test_metrics['loss'] = round(test_loss, 5)
print(json.dumps(test_metrics, indent=2))

{
  "accuracy": 0.8194,
  "precision_positive": 0.8391,
  "recall_positive": 0.7945,
  "recall_negative": 0.8447,
  "macro_f1": 0.8193,
  "roc_auc": 0.8894,
  "confusion_matrix": [
    [
      484,
      89
    ],
    [
      120,
      464
    ]
  ],
  "loss": 0.426
}


## 8. Export and inference
The final weights and vocabulary are exported for the ReviewLens website. The website performs the same preprocessing and neural-network calculations in the browser.

In [8]:
export_browser_model(model, vocabulary, MAX_LENGTH, test_metrics, Path('public/model'))

from predict import predict_review
for example in [
    'The product works perfectly and I love it.',
    'It stopped working after two days and the quality is poor.',
]:
    label, confidence = predict_review(example)
    print(label, f'({confidence:.1%} confidence)')

Positive review (100.0% confidence)
Negative review (98.9% confidence)


## 9. Ethics and limitations
The dataset is limited to older English Amazon reviews. Sarcasm, mixed sentiment, spelling variation and cultural context can reduce reliability. Model confidence is not certainty. The website processes review text locally and does not send it to an external service. The model should support human judgement rather than make high-impact decisions automatically.